# 📓 Session 8 · Exercises
### Classification with logistic regression

The lecture gave the 19-column table a label as its target: 1 when the next 20 days were more volatile than the last 20, 0 otherwise. A straight line fitted to that label gave probabilities below zero, the logistic curve did not, and the curve's probabilities were turned into predictions, scored with the four counts and the AUC, and improved with a penalty whose strength is called `C`.

These exercises do the same on the same table, one piece at a time, and then on every instrument in the data.

## How to use this notebook

- Run the **setup cell** below first. It loads the price data and the lecture's table, makes the label, and imports the scikit-learn pieces.
- Each exercise has a **task**, then a **code cell** for your work. Cells with `...` are blanks to fill in. Replace them with real code.
- Stuck? Open the **💡 Hint**, but only after a genuine attempt. Open the **✅ Solution** to *check* yourself, not to skip the thinking.
- Every cell runs cleanly even with the blanks still in place, so pressing **Run all** never floods you with errors.
- Most exercises stand alone. A few short runs build on each other (B4 to B7, C1 to C3, E2 to E4, F1 to F2, G5 to G6); the task says which earlier exercise it continues from. If one defeats you, open its solution, run it, and carry on. Section J uses the function from J1, and section K is five small cases that each start from scratch.

**You are not expected to finish all of these.** Do what you can, and come back to the rest when you revise. Short on time? Read the hint, then the solution. A worked solution you genuinely understand is real learning too.

**Returns are in percent here**, exactly as in the lecture. The label has no units at all: it is 1 or 0.

### Difficulty

| badge | what to expect |
|:--|:--|
| ★☆☆☆☆ | One step, straight from the lecture. You are checking that you can type it. |
| ★★☆☆☆ | The same idea on new data, or two steps in a row. Nothing to decide. |
| ★★★☆☆ | Combine two ideas, or adapt a pattern rather than copy it. |
| ★★★★☆ | You choose the approach. Several steps, and something has to be worked out before you type. |
| ★★★★★ | A genuine puzzle: an insight, or a constraint that rules out the obvious route. Always solvable with what you have. |

The stars rate the work against **this** session. A three-star task here assumes everything from Sessions 1 to 7, so it is a bigger piece of work than a three-star task in an earlier notebook.

Some exercises also carry a **revisits** tag. Those need something from an earlier session as well as today's material, and they are there on purpose: the skills are meant to accumulate.

## 🧰 Your toolkit for today

Everything from Sessions 1 to 7 still applies. This card holds what Session 8 added.

> Names in brackets (`frame`, `columns`, `model`, ...) are **placeholders**: put your own variable there. **Hover any tool** to see what it does.

<p style="line-height:2.1"><strong>The label and the model</strong><br>
<code style="cursor:help" title="A label from a comparison: True and False become 1 and 0.">(frame['a'] > frame['b']).astype(int)</code> &nbsp;&nbsp; <code style="cursor:help" title="Fits the log-odds of the label as a straight line in the columns. Create, then .fit(X, y), as for LinearRegression. A ridge penalty of strength C=1 is on by default.">LogisticRegression()</code> &nbsp;&nbsp; <code style="cursor:help" title="intercept_ holds one number in an array; coef_ has one ROW per class boundary, so coef_[0, 0] is the first slope.">model.intercept_[0]  ·  model.coef_[0, 0]</code> &nbsp;&nbsp; <code style="cursor:help" title="The classes in the order predict_proba uses for its columns: 0 then 1.">model.classes_</code> &nbsp;&nbsp; <code style="cursor:help" title="A coefficient adds to the log-odds; its exponential multiplies the odds.">np.exp(model.coef_)</code></p>

<p style="line-height:2.1"><strong>Probabilities and predictions</strong><br>
<code style="cursor:help" title="One row per observation, one column per class, each row summing to one. Column 1 is the probability of a 1.">model.predict_proba(X)[:, 1]</code> &nbsp;&nbsp; <code style="cursor:help" title="The 0/1 predictions: 1 where the probability of a 1 is at least one half.">model.predict(X)</code> &nbsp;&nbsp; <code style="cursor:help" title="Predictions at any other threshold: a comparison on the probability column.">(p >= 0.6).astype(int)</code></p>

<p style="line-height:2.1"><strong>How it is fitted</strong><br>
<code style="cursor:help" title="The objective, averaged over the rows: minus the log of the probability given to what happened. Takes the true labels and the probabilities.">log_loss(y, p)</code> &nbsp;&nbsp; <code style="cursor:help" title="How many steps the solver may take (default 100) and how many it took.">LogisticRegression(max_iter=1000)  ·  model.n_iter_</code></p>

<p style="line-height:2.1"><strong>Scoring a classifier</strong><br>
<code style="cursor:help" title="The share of predictions that match the label. True labels first, predictions second, like every metric.">accuracy_score(y, predicted)</code> &nbsp;&nbsp; <code style="cursor:help" title="A 2 by 2 array: rows are what happened, columns are the prediction, both in the order 0 then 1.">confusion_matrix(y, predicted)</code> &nbsp;&nbsp; <code style="cursor:help" title="Precision: the share of predicted 1s that were 1. Recall: the share of real 1s that were predicted.">precision_score(y, predicted)  ·  recall_score(y, predicted)</code> &nbsp;&nbsp; <code style="cursor:help" title="The false positive rate and recall at every threshold, plus the thresholds. Pass the PROBABILITIES.">roc_curve(y, p)</code> &nbsp;&nbsp; <code style="cursor:help" title="The area under the ROC curve: 0.5 for a random ranking, 1 for a perfect one. Pass the PROBABILITIES, never the 0/1 predictions.">roc_auc_score(y, p)</code></p>

<p style="line-height:2.1"><strong>Choosing, and the penalty</strong><br>
<code style="cursor:help" title="The same folds as for regression with a classification score. Larger is better, so no minus sign. Also 'accuracy'.">cross_val_score(model, X, y, cv=folds, scoring='roc_auc')</code> &nbsp;&nbsp; <code style="cursor:help" title="The strength of the penalty, upside down: C = 1/alpha, so a small C is a strong penalty. Standardise in a pipeline and search a grid in factors of ten.">LogisticRegression(C=0.01)</code> &nbsp;&nbsp; <code style="cursor:help" title="The lasso's charge on absolute values, which sets coefficients to exactly zero. Needs a solver that can handle it.">LogisticRegression(penalty='l1', solver='liblinear', C=0.01)</code> &nbsp;&nbsp; <code style="cursor:help" title="The grid key is the step name, two underscores, the argument.">{'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]}</code></p>

**Formulas you will reach for**

| what | formula |
|:--|:--|
| The sigmoid | $$\sigma(z)=\dfrac{1}{1+e^{-z}}$$ |
| Logistic regression | $$P(y=1\mid x)=\sigma(\beta_0+\beta_1 x)$$ |
| Log-odds | $$\log\dfrac{P}{1-P}=\beta_0+\beta_1 x$$ |
| Log-loss | $$-\dfrac{1}{n}\sum_i \big[y_i\log \hat p_i+(1-y_i)\log(1-\hat p_i)\big]$$ |
| Accuracy | $$\dfrac{TP+TN}{TP+TN+FP+FN}$$ |
| Precision, recall | $$\dfrac{TP}{TP+FP},\qquad \dfrac{TP}{TP+FN}$$ |
| False positive rate | $$\dfrac{FP}{FP+TN}$$ |
| The objective with the penalty | $$C\cdot\text{log-loss}+\tfrac{1}{2}\sum_j\beta_j^2$$ |


---

## ⚙️ Setup: run this first

This loads the price data and the lecture's 19-column table, makes the label, splits by date, and imports the scikit-learn pieces. If you are in Google Colab it downloads the data by itself.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             roc_curve, roc_auc_score, log_loss)
from sklearn.model_selection import cross_val_score, TimeSeriesSplit, GridSearchCV

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def load_csv(filename, **kwargs):
    """Read one of the course CSV files, wherever it happens to be."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    if REPO_RAW_URL is not None:
        return pd.read_csv(REPO_RAW_URL + filename, **kwargs)
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# Eleven instruments, 2015 to 2024. Returns in PERCENT, as in the lecture.
prices = load_csv("prices.csv", parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change().dropna() * 100

# The lecture's table: 19 columns looking back, vol_next looking forward
table = load_csv("market_features.csv", parse_dates=["date"]).set_index("date")
columns = list(table.columns[:-1])

# The lecture's label: 1 when the next 20 days were more volatile than the last 20
table["rising"] = (table["vol_next"] > table["vol_20d"]).astype(int)

train = table.loc[:"2022-12-31"]
test = table.loc["2023-01-01":]

folds = TimeSeriesSplit(n_splits=5)

print("table:", table.shape, "rows x columns")
print("train:", len(train), " test:", len(test), " feature columns:", len(columns))
print("share of days with a rise, train:", round(train["rising"].mean(), 3), " test:", round(test["rising"].mean(), 3))

---

## 🏷️ A · Making labels

A label is a column of ones and zeros that you make from a comparison. These exercises make the lecture's label and a few others, and find the share a model has to beat for each.

### A1 · The lecture's label, by hand  ★☆☆☆☆  · revisits S3

Make the label yourself as `own`: 1 where `vol_next` is larger than `vol_20d`, 0 otherwise. Then check it agrees with the setup's `table['rising']` on every row.

In [ ]:
own = ...
print(own)
print('same on every row:', ...)

<details>
<summary>💡 Hint 1</summary>

A comparison of two columns gives `True` and `False`; `.astype(int)` turns them into 1 and 0.

</details>

<details>
<summary>💡 Hint 2</summary>

`(own == table['rising']).all()` is `True` only when every row agrees.

</details>

<details>
<summary>✅ Solution</summary>

```python
own = (table['vol_next'] > table['vol_20d']).astype(int)
print(own.tail(3))
print('same on every row:', (own == table['rising']).all())
```

`True`. The comparison is asked of every row at once, and `.all()` is the vectorised way to ask whether a whole column of checks passed.

</details>

---

### A2 · The share to beat  ★★☆☆☆

Print the share of days with a rise in `train` and in `test`. Then print the accuracy of predicting 0 on every test day.

In [ ]:
print('train:', ...)
print('test :', ...)
print('predict 0 everywhere:', ...)

<details>
<summary>💡 Hint 1</summary>

The mean of a column of ones and zeros is the share of ones.

</details>

<details>
<summary>💡 Hint 2</summary>

Predicting 0 is right on every day whose label is 0, so its accuracy is `1 - test['rising'].mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('train:', round(train['rising'].mean(), 4))
print('test :', round(test['rising'].mean(), 4))
print('predict 0 everywhere:', round(1 - test['rising'].mean(), 4))
```

0.481 and 0.467, so predicting 0 on every test day is right 53.3% of the time. That is the majority rule, and every model in this notebook is measured against it.

</details>

---

### A3 · A label with a fixed cut  ★★☆☆☆  · revisits S3

Make a second label, `busy`: 1 where `vol_next` is above 1.0 (a daily volatility of one percent). Print its share in the training and test rows, and the majority rule's accuracy on the test rows.

In [ ]:
busy = ...
busy_train = ...
busy_test = ...

print(..., ...)
print('majority rule:', ...)

<details>
<summary>💡 Hint 1</summary>

Compare a column with a number this time: `table['vol_next'] > 1.0`.

</details>

<details>
<summary>💡 Hint 2</summary>

Slice the label by date, `busy.loc[:'2022-12-31']`, exactly as the setup sliced the table. The majority rule's accuracy is the larger of the share and one minus the share.

</details>

<details>
<summary>✅ Solution</summary>

```python
busy = (table['vol_next'] > 1.0).astype(int)
busy_train = busy.loc[:'2022-12-31']
busy_test = busy.loc['2023-01-01':]

print(round(busy_train.mean(), 3), round(busy_test.mean(), 3))
print('majority rule:', round(max(busy_test.mean(), 1 - busy_test.mean()), 3))
```

0.397 in the training rows and 0.122 in the test rows: 2023 and 2024 were calm, so the majority rule scores 87.8% on this label with no model at all. A fixed cut makes an unbalanced label, and the share to beat depends on which years are being scored.

</details>

---

### A4 · A rise with a margin  ★★☆☆☆

Make `rising_20`: 1 where `vol_next` is more than 1.2 times `vol_20d`, a rise of at least 20 percent. Print its share in the training rows next to the share of the plain `rising` label.

In [ ]:
rising_20 = ...
print(..., ...)

<details>
<summary>💡 Hint</summary>

Multiply the column before comparing: `table['vol_next'] > 1.2 * table['vol_20d']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
rising_20 = (table['vol_next'] > 1.2 * table['vol_20d']).astype(int)
print(round(rising_20.loc[:'2022-12-31'].mean(), 3), round(train['rising'].mean(), 3))
```

0.315 against 0.481. Demanding a margin makes a rise rarer. The same table can carry many labels, and each is a different question.

</details>

---

### A5 · A function that makes the label  ★★★☆☆  · revisits S2

Write `make_label(frame, margin)`: it returns a Series of ones and zeros that is 1 where `vol_next` is more than `margin` times `vol_20d`. Check that `make_label(table, 1.0)` agrees with `table['rising']` everywhere.

In [ ]:
def make_label(frame, margin):
    ...

check = make_label(table, 1.0)
print('agrees:', ...)

<details>
<summary>💡 Hint 1</summary>

The body is A4's line with `margin` in place of `1.2` and `frame` in place of `table`, then `return`.

</details>

<details>
<summary>💡 Hint 2</summary>

`(check == table['rising']).all()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def make_label(frame, margin):
    return (frame['vol_next'] > margin * frame['vol_20d']).astype(int)

check = make_label(table, 1.0)
print('agrees:', (check == table['rising']).all())
```

`True`. A margin of 1.0 is the plain comparison. The function is one line, and it is what the next exercise loops over.

</details>

---

### A6 · How the share falls with the margin  ★★★☆☆  · revisits S2

Using `make_label` from A5, loop over the margins `[1.0, 1.1, 1.2, 1.5, 2.0]` and store the share of ones in the **training** rows in a dictionary `share_by_margin`. Print it.

In [ ]:
share_by_margin = {}

for margin in [1.0, 1.1, 1.2, 1.5, 2.0]:
    ...

print(share_by_margin)

<details>
<summary>💡 Hint</summary>

Inside the loop: `share_by_margin[margin] = round(float(make_label(train, margin).mean()), 3)`. Calling the function on `train` gives the training share directly; `float()` makes it print as a plain number.

</details>

<details>
<summary>✅ Solution</summary>

```python
share_by_margin = {}

for margin in [1.0, 1.1, 1.2, 1.5, 2.0]:
    share_by_margin[margin] = round(float(make_label(train, margin).mean()), 3)

print(share_by_margin)
```

From 0.481 at a margin of 1.0 down to 0.095 at 2.0: a doubling of volatility in a month happens on one training day in 11. A loop over a function into a dictionary is the shape of every search in this course.

</details>

---

## 📈 B · The straight line, and the curve

Why a regression cannot give a probability, and the curve that can. B4 to B7 share the model B4 fits.

### B1 · A regression on the label  ★☆☆☆☆

Fit `LinearRegression` on `vol_20d` with `rising` as the target, and print the intercept and the slope.

In [ ]:
line = LinearRegression()
...
print(..., ...)

<details>
<summary>💡 Hint</summary>

`.fit(train[['vol_20d']], train['rising'])`. The target can be a column of ones and zeros; nothing stops it.

</details>

<details>
<summary>✅ Solution</summary>

```python
line = LinearRegression()
line.fit(train[['vol_20d']], train['rising'])
print(line.intercept_, line.coef_)
```

An intercept of 0.682 and a slope of -0.205. The higher this month's volatility, the lower the fitted value, which is the direction the data has: high volatility is followed by lower volatility more often than not.

</details>

---

### B2 · Where the line leaves the band  ★★☆☆☆  · revisits S3

Compute the fitted values of a regression on `vol_20d` for the training rows. Print the smallest and the largest, and count how many are below 0 or above 1.

In [ ]:
line = LinearRegression()
line.fit(train[['vol_20d']], train['rising'])
fitted = ...

print(..., ...)
print('outside [0, 1]:', ...)

<details>
<summary>💡 Hint 1</summary>

`line.predict(train[['vol_20d']])` is an array of fitted values.

</details>

<details>
<summary>💡 Hint 2</summary>

Two masks joined with `|`: `((fitted < 0) | (fitted > 1)).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
line = LinearRegression()
line.fit(train[['vol_20d']], train['rising'])
fitted = line.predict(train[['vol_20d']])

print(fitted.min(), fitted.max())
print('outside [0, 1]:', ((fitted < 0) | (fitted > 1)).sum())
```

From -0.52 to 0.64, with 28 training days below zero. Read as a probability, a negative number means nothing, and a straight line has no way to stop at the edge of the band.

</details>

---

### B3 · The sigmoid, as a function  ★★☆☆☆  · revisits S2

Write `sigmoid(z)` from the formula and print its value at −2, 0 and 2.

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

In [ ]:
def sigmoid(z):
    ...

print(sigmoid(-2), sigmoid(0), sigmoid(2))

<details>
<summary>💡 Hint</summary>

`np.exp(-z)` is $e^{-z}$. The function works on a single number and on a whole array alike.

</details>

<details>
<summary>✅ Solution</summary>

```python
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(sigmoid(-2), sigmoid(0), sigmoid(2))
```

0.119, 0.5 and 0.881. Any number in, a number between 0 and 1 out, and one half exactly at zero. Keep this function: B6 uses it.

</details>

---

### B4 · Logistic regression in three lines  ★☆☆☆☆

Fit `LogisticRegression` on `vol_20d` with `rising` as the target, and print the intercept and the coefficient.

In [ ]:
model = LogisticRegression()
...
print(..., ...)

<details>
<summary>💡 Hint</summary>

Create it, then `.fit(train[['vol_20d']], train['rising'])`. `coef_` has two pairs of brackets: one row of coefficients per boundary between classes.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
print(model.intercept_, model.coef_)
```

An intercept of 1.163 and a slope of -1.340, on the log-odds scale. The slope is negative, as the regression's was, but these two numbers now describe a curve that stays between 0 and 1.

</details>

---

### B5 · Where the curve crosses one half  ★★☆☆☆

Using `model` from B4, compute the volatility at which the probability of a rise is exactly one half, and print it. Then check by predicting the probability at that volatility.

$$\beta_0 + \beta_1 x = 0 \quad\Longrightarrow\quad x = -\beta_0 / \beta_1$$

In [ ]:
crossing = ...
print(crossing)

check = ...
print(check)

<details>
<summary>💡 Hint 1</summary>

`-model.intercept_[0] / model.coef_[0, 0]`: `[0]` takes the number out of the intercept array, `[0, 0]` takes the first entry of the first row of `coef_`.

</details>

<details>
<summary>💡 Hint 2</summary>

`predict_proba` wants a table with the same column name, so build a one-row DataFrame around the number: `model.predict_proba(pd.DataFrame({'vol_20d': [crossing]}))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
crossing = -model.intercept_[0] / model.coef_[0, 0]
print(crossing)

check = model.predict_proba(pd.DataFrame({'vol_20d': [crossing]}))
print(check)
```

0.868 percent, and the check returns `[[0.5, 0.5]]`. Below that volatility the model predicts a rise, above it no rise.

</details>

---

### B6 · The curve by hand, and by the model  ★★★☆☆  · revisits S2

For the volatilities 0.5, 1.0 and 2.0, compute the probability of a rise two ways: with your `sigmoid` from B3 applied to the score `intercept + coefficient * x`, and with `model.predict_proba` from B4. Print both, and check they agree to six decimals.

In [ ]:
vols = np.array([0.5, 1.0, 2.0])

by_hand = ...
by_model = ...

print(by_hand)
print(by_model)
print('agree:', ...)

<details>
<summary>💡 Hint 1</summary>

The score is `model.intercept_[0] + model.coef_[0, 0] * vols`, an array; `sigmoid` of it is an array of probabilities.

</details>

<details>
<summary>💡 Hint 2</summary>

`model.predict_proba(pd.DataFrame({'vol_20d': vols}))[:, 1]` is the second column. `np.abs(by_hand - by_model).max() < 1e-6` is the check.

</details>

<details>
<summary>✅ Solution</summary>

```python
vols = np.array([0.5, 1.0, 2.0])

by_hand = sigmoid(model.intercept_[0] + model.coef_[0, 0] * vols)
by_model = model.predict_proba(pd.DataFrame({'vol_20d': vols}))[:, 1]

print(by_hand)
print(by_model)
print('agree:', np.abs(by_hand - by_model).max() < 1e-6)
```

0.621, 0.456 and 0.180, and `True`. `predict_proba` is the sigmoid of the linear score and nothing more; the two numbers from B4 are the whole model.

</details>

---

### B7 · The odds, in a sentence  ★★☆☆☆  · revisits S1

Compute the factor by which one percentage point more volatility multiplies the odds of a rise, from `model` in B4, and print one sentence with an f-string that states it to two decimals.

In [ ]:
factor = ...
sentence = ...
print(sentence)

<details>
<summary>💡 Hint</summary>

`np.exp(model.coef_[0, 0])`. In the f-string, `{factor:.2f}` rounds.

</details>

<details>
<summary>✅ Solution</summary>

```python
factor = np.exp(model.coef_[0, 0])
sentence = f'One percentage point more volatility this month multiplies the odds of a rise next month by {factor:.2f}.'
print(sentence)
```

The factor is 0.26: the odds fall to about a quarter. A coefficient adds on the log-odds scale, so its exponential multiplies the odds, and that factor is the number to quote.

</details>

---

### B8 · The curve over the data  ★★★★☆  · revisits S3

Draw the fitted curve from B4 over the data it was fitted to. For each band of `vol_20d` in `bands`, compute the share of training days with a rise and the band's mean volatility with a mask; plot those as dots, and the curve as a line over `np.linspace(0.05, 4, 200)`.

In [ ]:
bands = [(0, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0), (1.0, 1.3), (1.3, 1.7), (1.7, 2.5), (2.5, 6)]
xs, shares = [], []
for low, high in bands:
    ...

grid = np.linspace(0.05, 4, 200)
curve = ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

The mask is `(train['vol_20d'] >= low) & (train['vol_20d'] < high)`; append `train['vol_20d'][mask].mean()` to `xs` and `train['rising'][mask].mean()` to `shares`.

</details>

<details>
<summary>💡 Hint 2</summary>

`curve = model.predict_proba(pd.DataFrame({'vol_20d': grid}))[:, 1]`, then `ax.scatter(xs, shares)` and `ax.plot(grid, curve)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
bands = [(0, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0), (1.0, 1.3), (1.3, 1.7), (1.7, 2.5), (2.5, 6)]
xs, shares = [], []
for low, high in bands:
    mask = (train['vol_20d'] >= low) & (train['vol_20d'] < high)
    xs.append(train['vol_20d'][mask].mean())
    shares.append(train['rising'][mask].mean())

grid = np.linspace(0.05, 4, 200)
curve = model.predict_proba(pd.DataFrame({'vol_20d': grid}))[:, 1]

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.scatter(xs, shares, label='share of days with a rise, per band')
ax.plot(grid, curve, label='logistic regression')
ax.set_xlabel('volatility over the last 20 days (%)')
ax.set_ylabel('probability of a rise')
ax.legend()
plt.show()
```

The dots fall from 0.89 in the calmest band to 0.11 above 1.7 percent, and the curve runs through them. The band shares are the data's own answer to the question the model is asked, which is why they are the right thing to draw the curve against.

</details>

---

## 🎲 C · Probabilities and predictions

What `predict_proba` returns, how `predict` reads it, and the mistake that costs the most. C1 to C3 share the model C1 fits.

### C1 · Two columns per day  ★☆☆☆☆

Fit the one-column logistic regression as `model` and call `predict_proba` on the test rows. Print `classes_`, the shape of the result, and its first three rows.

In [ ]:
model = LogisticRegression()
...
probs = ...

print(...)
print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`probs = model.predict_proba(test[['vol_20d']])`. `.shape` is a pair of numbers; `probs[:3]` is the first three rows.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
probs = model.predict_proba(test[['vol_20d']])

print(model.classes_)
print(probs.shape)
print(probs[:3])
```

`[0 1]`, `(482, 2)`, and three rows that each add up to one. The columns come in the order of `classes_`, so the second column is the probability of a rise.

</details>

---

### C2 · predict, written out  ★★☆☆☆  · revisits S3

Take the second column of `probs` from C1 as `p`. Build the 0/1 predictions two ways, with `model.predict` and with a comparison against 0.5, and write an assertion that the two arrays are equal on every day.

In [ ]:
p = ...
predicted = ...
by_hand = ...

assert ...
print('share predicted as a rise:', ...)

<details>
<summary>💡 Hint 1</summary>

`probs[:, 1]` is every row of the second column.

</details>

<details>
<summary>💡 Hint 2</summary>

`(p >= 0.5).astype(int)` for the comparison; `(predicted == by_hand).all()` in the assertion; `predicted.mean()` is the share of ones.

</details>

<details>
<summary>✅ Solution</summary>

```python
p = probs[:, 1]
predicted = model.predict(test[['vol_20d']])
by_hand = (p >= 0.5).astype(int)

assert (predicted == by_hand).all()
print('share predicted as a rise:', predicted.mean())
```

The assertion passes, and the model predicts a rise on 66.2% of the test days. `.predict()` is a comparison with one half, and once that is written out the threshold is yours to change.

</details>

---

### C3 · A stricter threshold  ★★☆☆☆

Using `p` from C2, count the test days on which the probability of a rise is at least 0.6, and the days on which it is at least 0.7.

In [ ]:
print('at least 0.6:', ...)
print('at least 0.7:', ...)

<details>
<summary>💡 Hint</summary>

`(p >= 0.6).sum()` counts the days that pass a comparison.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('at least 0.6:', (p >= 0.6).sum())
print('at least 0.7:', (p >= 0.7).sum())
```

51 days at 0.6, and none at 0.7: the model never gives a test day a probability above 0.67. A threshold above that predicts no rise at all.

</details>

---

### C4 · Fix the bug: the wrong column  ★★★☆☆  · revisits S1

The cell below scores the model with `roc_auc_score` and gets a number far below one half, which would mean a model worse than a random ranking. Find the mistake, fix it, and print the right AUC.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])

wrong = roc_auc_score(test['rising'], model.predict_proba(test[['vol_20d']])[:, 0])
print('as written:', wrong)

right = ...
print('fixed     :', right)

<details>
<summary>💡 Hint 1</summary>

Column 0 of `predict_proba` is the probability of **no** rise. Scoring the label 1 with the probability of 0 ranks every day backwards.

</details>

<details>
<summary>💡 Hint 2</summary>

`[:, 1]` is the column that goes with the label.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])

wrong = roc_auc_score(test['rising'], model.predict_proba(test[['vol_20d']])[:, 0])
print('as written:', wrong)

right = roc_auc_score(test['rising'], model.predict_proba(test[['vol_20d']])[:, 1])
print('fixed     :', right)
```

0.193 as written and 0.807 fixed, and the two add up to one: the wrong column is the right ranking turned upside down. An AUC well below 0.5 is almost always this bug, never a model that is worse than random.

</details>

---

## 🚶 D · How the curve is fitted

The objective, the steps, and the warning.

### D1 · Log-loss, of the model and of a guess  ★★☆☆☆

Fit the one-column model, and print its log-loss on the training rows next to the log-loss of giving every training day a probability of one half.

In [ ]:
model = LogisticRegression()
...
p_train = ...

print('the model  :', ...)
print('one half   :', ...)

<details>
<summary>💡 Hint 1</summary>

`log_loss(train['rising'], p_train)` with the probability column.

</details>

<details>
<summary>💡 Hint 2</summary>

A probability of one half for every day is `np.full(len(train), 0.5)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p_train = model.predict_proba(train[['vol_20d']])[:, 1]

print('the model  :', log_loss(train['rising'], p_train))
print('one half   :', log_loss(train['rising'], np.full(len(train), 0.5)))
```

0.6382 against 0.6931, which is $\log 2$: the cost of saying one half about everything. The fitted curve is the pair of numbers with the lowest log-loss on these rows, and this is how much lower it got.

</details>

---

### D2 · Watch the steps  ★★☆☆☆

Fit the one-column model with `max_iter` set to 1, 2, 3 and 10 in a loop. For each, print the limit, `n_iter_`, the coefficient and the training log-loss.

In [ ]:
for n in [1, 2, 3, 10]:
    steps = LogisticRegression(max_iter=n)
    ...

<details>
<summary>💡 Hint</summary>

After fitting: `steps.n_iter_`, `steps.coef_.round(3)`, and `log_loss` of `steps.predict_proba(train[['vol_20d']])[:, 1]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for n in [1, 2, 3, 10]:
    steps = LogisticRegression(max_iter=n)
    steps.fit(train[['vol_20d']], train['rising'])
    p_n = steps.predict_proba(train[['vol_20d']])[:, 1]
    print(n, steps.n_iter_, steps.coef_.round(3), round(log_loss(train['rising'], p_n), 5))
```

The log-loss falls from 0.6766 after one step to 0.6382, and the run allowed 10 steps stops after 7, because the coefficients had stopped changing. Each fit with a small limit shows the solver part of the way down the hill.

</details>

---

### D3 · Log-loss from the formula  ★★★☆☆  · revisits S3

Compute the training log-loss of the one-column model yourself, from the formula, without `log_loss`, and check it against the function.

$$-\frac{1}{n}\sum_i \big[y_i \log \hat p_i + (1 - y_i)\log(1 - \hat p_i)\big]$$

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p_train = model.predict_proba(train[['vol_20d']])[:, 1]
y = train['rising'].values

by_hand = ...
print(by_hand)
print(log_loss(y, p_train))

<details>
<summary>💡 Hint 1</summary>

With `y` as ones and zeros, `y * np.log(p_train)` keeps the log of the probability on the days with a rise and `(1 - y) * np.log(1 - p_train)` on the others. Add them, take the mean, put a minus in front.

</details>

<details>
<summary>💡 Hint 2</summary>

The whole thing is one line of array arithmetic: no loop over days.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p_train = model.predict_proba(train[['vol_20d']])[:, 1]
y = train['rising'].values

by_hand = -(y * np.log(p_train) + (1 - y) * np.log(1 - p_train)).mean()
print(by_hand)
print(log_loss(y, p_train))
```

0.63825 both ways. Multiplying by `y` and by `1 - y` is the switch in the formula written as arithmetic, and it is vectorised over all 1,894 days at once.

</details>

---

### D4 · Provoke the warning, then cure it  ★★★☆☆  · revisits S6

Fit `LogisticRegression(max_iter=5)` on all 19 **raw** columns and watch the `ConvergenceWarning` appear. Then fit again with enough steps for it to go away, print both test AUCs, and print how many steps the second fit took.

In [ ]:
few = LogisticRegression(max_iter=5)
few.fit(train[columns], train['rising'])
print('5 steps   :', roc_auc_score(test['rising'], few.predict_proba(test[columns])[:, 1]))

enough = ...
...
print('more steps:', ...)
print('steps taken:', ...)

<details>
<summary>💡 Hint</summary>

`LogisticRegression(max_iter=1000)` is enough; `enough.n_iter_` says how many it used.

</details>

<details>
<summary>✅ Solution</summary>

```python
few = LogisticRegression(max_iter=5)
few.fit(train[columns], train['rising'])
print('5 steps   :', roc_auc_score(test['rising'], few.predict_proba(test[columns])[:, 1]))

enough = LogisticRegression(max_iter=1000)
enough.fit(train[columns], train['rising'])
print('more steps:', roc_auc_score(test['rising'], enough.predict_proba(test[columns])[:, 1]))
print('steps taken:', enough.n_iter_)
```

The first fit prints a warning and scores 0.636; the second is silent, takes 80 steps and scores 0.809. On raw columns the solver needs 80 steps; on standardised ones it needs 31, which is the other cure the warning names.

</details>

---

## 🧮 E · Scoring with the four counts

Accuracy, the confusion matrix, precision and recall, each first by hand and then with the function. E2 to E4 share the masks E2 builds.

### E1 · Accuracy, two ways  ★★☆☆☆  · revisits S3

Fit the one-column model, predict the test rows, and compute the accuracy as the share of days where the prediction equals the label, then with `accuracy_score`. Print both next to the majority rule from A2.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = ...

print('by hand   :', ...)
print('function  :', ...)
print('majority  :', ...)

<details>
<summary>💡 Hint</summary>

`(predicted == test['rising']).mean()` and `accuracy_score(test['rising'], predicted)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

print('by hand   :', (predicted == test['rising']).mean())
print('function  :', accuracy_score(test['rising'], predicted))
print('majority  :', 1 - test['rising'].mean())
```

0.6929 both ways, against 0.5332 for predicting 0 on every day. Accuracy only means something next to that share.

</details>

---

### E2 · The four counts, with masks  ★★☆☆☆  · revisits S4

Build two masks, `rose` (the label is 1) and `pred_rise` (the prediction is 1), and count the four kinds of test day: predicted rise and it rose, predicted rise and it did not, predicted no rise but it rose, predicted no rise and none came.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

rose = ...
pred_rise = ...

tp = ...
fp = ...
fn = ...
tn = ...
print(tp, fp, fn, tn)

<details>
<summary>💡 Hint 1</summary>

`rose = test['rising'].values == 1` and `pred_rise = predicted == 1` are boolean arrays.

</details>

<details>
<summary>💡 Hint 2</summary>

`(pred_rise & rose).sum()`, `(pred_rise & ~rose).sum()`, `(~pred_rise & rose).sum()`, `(~pred_rise & ~rose).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

rose = test['rising'].values == 1
pred_rise = predicted == 1

tp = (pred_rise & rose).sum()
fp = (pred_rise & ~rose).sum()
fn = (~pred_rise & rose).sum()
tn = (~pred_rise & ~rose).sum()
print(tp, fp, fn, tn)
```

198, 121, 27 and 136, which add up to 482. The four counts are the whole story of a classifier at one threshold; every score in this section is a ratio of some of them.

</details>

---

### E3 · confusion_matrix  ★★☆☆☆

Print `confusion_matrix` for the same predictions, and check that its four entries are the four counts from E2 in the layout the lecture described: `tn` top left, `tp` bottom right.

In [ ]:
cm = ...
print(cm)
print('matches:', ...)

<details>
<summary>💡 Hint</summary>

`confusion_matrix(test['rising'], predicted)`. `cm[0, 0]` is top left and `cm[1, 1]` bottom right; compare them with `tn` and `tp`.

</details>

<details>
<summary>✅ Solution</summary>

```python
cm = confusion_matrix(test['rising'], predicted)
print(cm)
print('matches:', cm[0, 0] == tn and cm[1, 1] == tp and cm[0, 1] == fp and cm[1, 0] == fn)
```

`True`. Rows are what happened and columns are the prediction, both with 0 first. Getting that layout wrong swaps precision and recall silently, which is why the check is worth a line.

</details>

---

### E4 · Precision and recall, two ways  ★★★☆☆  · revisits S4

Compute precision and recall from the counts `tp`, `fp` and `fn` of E2, then with `precision_score` and `recall_score`, and print all four numbers.

In [ ]:
print('precision by hand:', ...)
print('precision function:', ...)
print('recall by hand   :', ...)
print('recall function   :', ...)

<details>
<summary>💡 Hint 1</summary>

Precision is `tp / (tp + fp)`, the share of predicted rises that happened. Recall is `tp / (tp + fn)`, the share of the rises that were predicted.

</details>

<details>
<summary>💡 Hint 2</summary>

The functions take the true labels first and the predictions second.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('precision by hand:', tp / (tp + fp))
print('precision function:', precision_score(test['rising'], predicted))
print('recall by hand   :', tp / (tp + fn))
print('recall function   :', recall_score(test['rising'], predicted))
```

Precision 0.621 and recall 0.880, both ways. The model predicts a rise on 319 days and is right on 198 of them, and it predicts 198 of the 225 rises that came.

</details>

---

### E5 · A function for the four counts  ★★★☆☆  · revisits S2

Write `four_counts(y, predicted)`: it returns a dictionary with the keys `'tp'`, `'fp'`, `'fn'` and `'tn'`. Run it on the test labels and the predictions of a one-column model, and print the dictionary.

In [ ]:
def four_counts(y, predicted):
    ...

model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
counts = four_counts(test['rising'].values, model.predict(test[['vol_20d']]))
print(counts)

<details>
<summary>💡 Hint 1</summary>

Inside, build the two masks from the arguments, `y == 1` and `predicted == 1`, then the four sums as in E2.

</details>

<details>
<summary>💡 Hint 2</summary>

Wrap each sum in `int()` so the dictionary prints plain numbers.

</details>

<details>
<summary>✅ Solution</summary>

```python
def four_counts(y, predicted):
    rose = y == 1
    pred_rise = predicted == 1
    return {'tp': int((pred_rise & rose).sum()), 'fp': int((pred_rise & ~rose).sum()),
            'fn': int((~pred_rise & rose).sum()), 'tn': int((~pred_rise & ~rose).sum())}

model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
counts = four_counts(test['rising'].values, model.predict(test[['vol_20d']]))
print(counts)
```

`{'tp': 198, 'fp': 121, 'fn': 27, 'tn': 136}`. Four lines that any classifier at any threshold can be handed, and the next exercise hands them several.

</details>

---

### E6 · Accuracy at six thresholds  ★★★☆☆  · revisits S2

Using `four_counts` from E5, loop over the thresholds `[0.4, 0.45, 0.5, 0.55, 0.6, 0.65]`: turn the test probabilities into predictions at each, compute the accuracy from the four counts, and store it in a dictionary. Print the dictionary and the best threshold.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]

acc_by_threshold = {}
for threshold in [0.4, 0.45, 0.5, 0.55, 0.6, 0.65]:
    ...

print(acc_by_threshold)
print('best:', ...)

<details>
<summary>💡 Hint 1</summary>

`predicted = (p >= threshold).astype(int)`, then `c = four_counts(...)` and accuracy is `(c['tp'] + c['tn']) / len(test)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`max(acc_by_threshold, key=acc_by_threshold.get)` is the key with the largest value.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]

acc_by_threshold = {}
for threshold in [0.4, 0.45, 0.5, 0.55, 0.6, 0.65]:
    predicted = (p >= threshold).astype(int)
    c = four_counts(test['rising'].values, predicted)
    acc_by_threshold[threshold] = round((c['tp'] + c['tn']) / len(test), 3)

print(acc_by_threshold)
print('best:', max(acc_by_threshold, key=acc_by_threshold.get))
```

Best at 0.55, with 0.743 against 0.693 at one half. A threshold is a setting, and here it was read off the test rows, which is not allowed for choosing; F6 does it on the training rows.

</details>

---

### E7 · The majority rule's four counts  ★★★☆☆  · revisits S4

Score the rule that predicts 0 on every test day with `confusion_matrix`, `accuracy_score` and `recall_score`. What does its confusion matrix look like, and why is its recall what it is?

In [ ]:
never = np.zeros(len(test), dtype=int)

print(...)
print('accuracy:', ...)
print('recall  :', ...)

<details>
<summary>💡 Hint</summary>

`np.zeros(len(test), dtype=int)` is a prediction of 0 on every day. The three functions take it like any other prediction.

</details>

<details>
<summary>✅ Solution</summary>

```python
never = np.zeros(len(test), dtype=int)

print(confusion_matrix(test['rising'], never))
print('accuracy:', accuracy_score(test['rising'], never))
print('recall  :', recall_score(test['rising'], never))
```

The right-hand column of the matrix is all zeros: 257 true negatives, 225 misses, nothing predicted. Accuracy 0.533 and recall 0: a rule that never predicts a rise cannot catch one. On this balanced label the model beats it on both; on a rare label accuracy alone would not show the difference.

</details>

---

## 📉 F · Thresholds and the ROC curve

Every threshold is one point; all of them are the curve. F1 and F2 share the probabilities F1 computes.

### F1 · Three thresholds, two shares  ★★☆☆☆

Fit the one-column model and take the test probabilities as `p`. For the thresholds 0.4, 0.5 and 0.6, print the recall and the false positive rate: the share of days **without** a rise on which a rise was predicted.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

for threshold in [0.4, 0.5, 0.6]:
    ...

<details>
<summary>💡 Hint 1</summary>

`pred = p >= threshold` is a mask. Recall is `(pred & rose).sum() / rose.sum()`.

</details>

<details>
<summary>💡 Hint 2</summary>

The false positive rate is `(pred & ~rose).sum() / (~rose).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

for threshold in [0.4, 0.5, 0.6]:
    pred = p >= threshold
    recall = (pred & rose).sum() / rose.sum()
    false_alarm_rate = (pred & ~rose).sum() / (~rose).sum()
    print(threshold, round(recall, 3), round(false_alarm_rate, 3))
```

At 0.4, recall 1.000 and a false positive rate of 0.895; at 0.6, 0.187 and 0.035. Raising the threshold lowers both. Each pair is one point of the ROC curve.

</details>

---

### F2 · The ROC curve, by hand and by the function  ★★★★☆  · revisits S3

Repeat F1 over `np.linspace(0, 1, 101)` and collect the two shares in two lists. Then call `roc_curve` on `p` and draw both on one figure: your points as dots, the function's curve as a line, false positive rate across and recall up.

In [ ]:
recalls, false_alarm_rates = [], []
for threshold in np.linspace(0, 1, 101):
    ...

fpr, tpr, thresholds = ..., ..., ...

fig, ax = plt.subplots(figsize=(5, 4))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

The loop body is F1's with `.append` in place of `print`.

</details>

<details>
<summary>💡 Hint 2</summary>

`fpr, tpr, thresholds = roc_curve(test['rising'], p)` returns three arrays. `ax.scatter(false_alarm_rates, recalls)` and `ax.plot(fpr, tpr)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
recalls, false_alarm_rates = [], []
for threshold in np.linspace(0, 1, 101):
    pred = p >= threshold
    recalls.append((pred & rose).sum() / rose.sum())
    false_alarm_rates.append((pred & ~rose).sum() / (~rose).sum())

fpr, tpr, thresholds = roc_curve(test['rising'], p)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(false_alarm_rates, recalls, s=12, label='101 thresholds by hand')
ax.plot(fpr, tpr, label='roc_curve')
ax.plot([0, 1], [0, 1], linestyle='--', color='grey')
ax.set_xlabel('false positive rate')
ax.set_ylabel('recall')
ax.legend()
plt.show()
```

The dots sit on the line. `roc_curve` used 158 thresholds, one at every probability where a point moves, and the loop used 101 evenly spaced ones; both draw the same curve, because the curve only depends on how the probabilities rank the days.

</details>

---

### F3 · The area  ★☆☆☆☆

Fit the one-column model and print its AUC on the test rows.

In [ ]:
model = LogisticRegression()
...
print(...)

<details>
<summary>💡 Hint</summary>

`roc_auc_score(test['rising'], model.predict_proba(test[['vol_20d']])[:, 1])`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
print(roc_auc_score(test['rising'], model.predict_proba(test[['vol_20d']])[:, 1]))
```

0.8075. The second argument is the probability column, not the 0/1 predictions.

</details>

---

### F4 · The AUC of a rule with no ranking  ★★☆☆☆

Compute the AUC of a prediction that gives every test day the same probability, 0.5, and of one that gives every day a probability of 0. Explain the two numbers in one sentence.

In [ ]:
same = np.full(len(test), 0.5)
zeros = ...

print(roc_auc_score(test['rising'], same))
print(...)

<details>
<summary>💡 Hint</summary>

`np.zeros(len(test))`. When every day has the same score there is no ranking, and the area is one half whatever the score is.

</details>

<details>
<summary>✅ Solution</summary>

```python
same = np.full(len(test), 0.5)
zeros = np.zeros(len(test))

print(roc_auc_score(test['rising'], same))
print(roc_auc_score(test['rising'], zeros))
```

0.5 both times. The AUC scores a ranking, and a rule that ranks nothing scores one half, which is why one half, not zero, is the floor.

</details>

---

### F5 · Precision and recall against the threshold  ★★★☆☆  · revisits S3

For thresholds from 0.3 to 0.7 in steps of 0.01, collect the precision and the recall of the one-column model's test predictions in two lists, and draw both against the threshold on one axis with a legend.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]

thresholds = np.arange(0.3, 0.7, 0.01)
precisions, recalls = [], []
for threshold in thresholds:
    ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`predicted = (p >= threshold).astype(int)`, then `precision_score(test['rising'], predicted)` and `recall_score(...)`, each appended.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(thresholds, precisions, label='precision')` and the same for recall.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]

thresholds = np.arange(0.3, 0.7, 0.01)
precisions, recalls = [], []
for threshold in thresholds:
    predicted = (p >= threshold).astype(int)
    precisions.append(precision_score(test['rising'], predicted))
    recalls.append(recall_score(test['rising'], predicted))

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(thresholds, precisions, label='precision')
ax.plot(thresholds, recalls, label='recall')
ax.set_xlabel('threshold')
ax.legend()
plt.show()
```

Recall falls from 1 to 0 as the threshold rises, and precision climbs. The two lines cross near 0.55; where a threshold should sit depends on which of the two errors costs more, which the next session takes up.

</details>

---

### F6 · Choose the threshold on the training rows  ★★★★☆  · revisits S5

A threshold is a setting, so it is chosen without the test rows. Compute the one-column model's probabilities on the **training** rows, find the threshold in `np.linspace(0.3, 0.7, 41)` with the highest training accuracy, and then print the test accuracy at that threshold.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p_train = ...
p_test = ...

best_threshold = None
best_accuracy = 0
for threshold in np.linspace(0.3, 0.7, 41):
    ...

print('chosen on train:', best_threshold, round(best_accuracy, 3))
print('test accuracy  :', ...)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, compute the training accuracy at the threshold and, if it beats `best_accuracy`, store both.

</details>

<details>
<summary>💡 Hint 2</summary>

Round the threshold, `round(threshold, 2)`, when you store it; `np.linspace` produces numbers like 0.4100000001.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p_train = model.predict_proba(train[['vol_20d']])[:, 1]
p_test = model.predict_proba(test[['vol_20d']])[:, 1]

best_threshold = None
best_accuracy = 0
for threshold in np.linspace(0.3, 0.7, 41):
    accuracy = accuracy_score(train['rising'], (p_train >= threshold).astype(int))
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_threshold = round(threshold, 2)

print('chosen on train:', best_threshold, round(best_accuracy, 3))
print('test accuracy  :', accuracy_score(test['rising'], (p_test >= best_threshold).astype(int)))
```

The training rows choose 0.57, at a training accuracy of 0.652, and that threshold scores 0.722 on the test rows. E6 found 0.55 by looking at the test rows, which is the difference between choosing and reporting.

</details>

---

### F7 · The AUC as a share of pairs  ★★★★★  · revisits S3

The lecture read the AUC as a probability: take a day with a rise and a day without at random, and the model ranks them correctly that often. Compute it that way. Split the test probabilities into those of the days with a rise and those without, count the pairs in which the rise has the higher probability, and divide by the number of pairs. Compare with `roc_auc_score`.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

p_rise = ...
p_calm = ...

wins = 0
...

print('by pairs:', ...)
print('function:', ...)

<details>
<summary>💡 Hint 1</summary>

`p[rose]` and `p[~rose]` split the probabilities.

</details>

<details>
<summary>💡 Hint 2</summary>

Loop `for value in p_rise:`; inside, `(value > p_calm).sum()` counts the no-rise days that one rise day beats. Add that up over every `value`, then divide by `len(p_rise) * len(p_calm)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

p_rise = p[rose]
p_calm = p[~rose]

wins = 0
for value in p_rise:
    wins = wins + (value > p_calm).sum()

print('by pairs:', wins / (len(p_rise) * len(p_calm)))
print('function:', roc_auc_score(test['rising'], p))
```

0.8075 by pairs and 0.8075 from the function, over 57,825 pairs; the tiny gap is pairs with exactly equal probabilities, which the function counts as half a win. One loop over the rise days and a vectorised comparison inside it is enough. The definition and the area are the same number.

</details>

---

## 🔄 G · Cross-validation, and C

The folds with a classification score, all 19 columns, and the strength of the penalty. G5 and G6 share the search G5 fits.

### G1 · AUC on the folds  ★☆☆☆☆

Cross-validate the one-column `LogisticRegression()` with `folds` and `scoring='roc_auc'`. Print the five scores and their mean.

In [ ]:
scores = ...
print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')`. Larger is better, so no minus sign.

</details>

<details>
<summary>✅ Solution</summary>

```python
scores = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'],
                         cv=folds, scoring='roc_auc')
print(scores.round(3))
print(scores.mean())
```

A mean of 0.783, with the folds from 0.70 to 0.85. The same five folds as for regression, read with a score that has no minus sign to strip.

</details>

---

### G2 · Accuracy on the folds  ★★☆☆☆  · revisits S5

Run the same cross-validation with `scoring='accuracy'` and print the mean next to the AUC's from G1. Why can one of them be far below the other?

In [ ]:
auc = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')
acc = ...
print('accuracy:', ...)
print('AUC     :', ...)

<details>
<summary>💡 Hint</summary>

Only the scoring string changes. Accuracy needs a threshold, which is one half inside `cross_val_score`; the AUC does not.

</details>

<details>
<summary>✅ Solution</summary>

```python
auc = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')
acc = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'],
                      cv=folds, scoring='accuracy')
print('accuracy:', acc.mean())
print('AUC     :', auc.mean())
```

0.647 against 0.783. The AUC scores the ranking of every fold's days; accuracy scores the predictions at one half, and on the first fold it is 0.498, since that fold's calm years sit almost entirely on one side of the threshold.

</details>

---

### G3 · All 19 columns, in a pipeline  ★★☆☆☆

Build a pipeline with a `StandardScaler` step `'scale'` and a `LogisticRegression(max_iter=1000)` step `'logit'`, cross-validate it on all `columns` with the AUC, and print the mean next to the one-column mean, which the first line computes.

In [ ]:
one_scores = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')

wide_pipe = Pipeline([...])
wide_scores = ...

print('19 columns:', ...)
print('one column:', ...)

<details>
<summary>💡 Hint</summary>

`Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])`, then `cross_val_score` on `train[columns]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
one_scores = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')

wide_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])
wide_scores = cross_val_score(wide_pipe, train[columns], train['rising'], cv=folds, scoring='roc_auc')

print('19 columns:', wide_scores.mean())
print('one column:', one_scores.mean())
```

0.679 against 0.783: on the folds, 19 columns rank the days worse than one. The same overfitting as in regression, with a different score reporting it.

</details>

---

### G4 · A loop over C  ★★★☆☆  · revisits S2

For each `C` in `[0.0001, 0.001, 0.01, 0.1, 1, 10]`, build the pipeline with that `C`, cross-validate it with the AUC, and store the mean in a dictionary `cv_by_C`. Print it and the best key.

In [ ]:
cv_by_C = {}

for C in [0.0001, 0.001, 0.01, 0.1, 1, 10]:
    ...

print(cv_by_C)
print('best:', ...)

<details>
<summary>💡 Hint 1</summary>

`LogisticRegression(C=C, max_iter=1000)` inside the pipeline; give the pipeline its own name so `wide_pipe` is kept.

</details>

<details>
<summary>💡 Hint 2</summary>

`max(cv_by_C, key=cv_by_C.get)`: the AUC is larger-is-better, so `max`.

</details>

<details>
<summary>✅ Solution</summary>

```python
cv_by_C = {}

for C in [0.0001, 0.001, 0.01, 0.1, 1, 10]:
    candidate = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=C, max_iter=1000))])
    candidate_scores = cross_val_score(candidate, train[columns], train['rising'], cv=folds, scoring='roc_auc')
    cv_by_C[C] = round(float(candidate_scores.mean()), 4)

print(cv_by_C)
print('best:', max(cv_by_C, key=cv_by_C.get))
```

Best at 0.0001, with the AUC rising from 0.676 at C = 10 to 0.732. A small C is a strong penalty, so the folds prefer the coefficients pulled hard towards zero. Note `max`, where the alpha loop used `min`.

</details>

---

### G5 · GridSearchCV over C  ★★☆☆☆

Let `GridSearchCV` run G4: the pipeline with `LogisticRegression(max_iter=1000)` and no `C`, the grid `{'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]}`, `folds`, and `scoring='roc_auc'`. Fit it as `search` and print the best C and score.

In [ ]:
base = ...
grid = ...

search = ...
...

print(...)
print(...)

<details>
<summary>💡 Hint</summary>

`GridSearchCV(base, grid, cv=folds, scoring='roc_auc')`, then `.fit`. The best score is positive this time.

</details>

<details>
<summary>✅ Solution</summary>

```python
base = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))])
grid = {'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]}

search = GridSearchCV(base, grid, cv=folds, scoring='roc_auc')
search.fit(train[columns], train['rising'])

print(search.best_params_)
print(search.best_score_)
```

C = 0.0001 at 0.7319, the numbers from your loop. `'logit__C'` is the step name, two underscores, then the argument.

</details>

---

### G6 · The test rows, once  ★★☆☆☆  · revisits S6

Use `search` from G5 to compute the test AUC, and read the chosen `C` back out of `search.best_estimator_`.

In [ ]:
print('test AUC:', ...)
print('C       :', ...)

<details>
<summary>💡 Hint 1</summary>

`search.predict_proba(test[columns])[:, 1]` uses the best pipeline, refitted on all the training rows.

</details>

<details>
<summary>💡 Hint 2</summary>

`search.best_estimator_.named_steps['logit'].C`.

</details>

<details>
<summary>✅ Solution</summary>

```python
print('test AUC:', roc_auc_score(test['rising'], search.predict_proba(test[columns])[:, 1]))
print('C       :', search.best_estimator_.named_steps['logit'].C)
```

0.8114 with C = 0.0001. The search chose C on the folds and refitted on every training row; the test rows are opened once, here.

</details>

---

### G7 · The same grid, scored by accuracy  ★★★☆☆

Run G5's grid search again with `scoring='accuracy'` and print the best C and score. Then print the AUC-chosen and accuracy-chosen C side by side.

In [ ]:
acc_search = ...
...

print(...)
print('chosen by AUC     :', ...)
print('chosen by accuracy:', ...)

<details>
<summary>💡 Hint</summary>

Only the scoring string changes. Read the two winners from the two `best_params_` dictionaries with the key `'logit__C'`.

</details>

<details>
<summary>✅ Solution</summary>

```python
acc_search = GridSearchCV(base, grid, cv=folds, scoring='accuracy')
acc_search.fit(train[columns], train['rising'])

print(acc_search.best_params_, acc_search.best_score_)
print('chosen by AUC     :', search.best_params_['logit__C'])
print('chosen by accuracy:', acc_search.best_params_['logit__C'])
```

Accuracy picks C = 0.01 at 0.609; the AUC picked 0.0001. The score is part of the choice, so it is fixed before the search and matched to what will be reported.

</details>

---

### G8 · The validation curve  ★★★☆☆  · revisits S6

Repeat G4 on the finer grid `np.logspace(-5, 2, 15)` and draw the mean AUC against C with a logarithmic x-axis. Add a horizontal line at the one-column mean, which the first line computes.

In [ ]:
one_scores = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')

Cs = np.logspace(-5, 2, 15)
aucs = []

for C in Cs:
    ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

The loop body is G4's, appending the mean to `aucs`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.plot(Cs, aucs, marker='o')`, `ax.set_xscale('log')`, and `ax.axhline(one_scores.mean(), linestyle='--')` for the one-column line.

</details>

<details>
<summary>✅ Solution</summary>

```python
one_scores = cross_val_score(LogisticRegression(), train[['vol_20d']], train['rising'], cv=folds, scoring='roc_auc')

Cs = np.logspace(-5, 2, 15)
aucs = []

for C in Cs:
    candidate = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=C, max_iter=1000))])
    candidate_scores = cross_val_score(candidate, train[columns], train['rising'], cv=folds, scoring='roc_auc')
    aucs.append(candidate_scores.mean())

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(Cs, aucs, marker='o', label='19 columns')
ax.axhline(one_scores.mean(), linestyle='--', color='grey', label='one column')
ax.set_xscale('log')
ax.set_xlabel('C')
ax.set_ylabel('cross-validated AUC')
ax.legend()
plt.show()
```

The curve rises from about 0.675 on the right to about 0.732 on the left and then flattens, and it never reaches the dashed line. For ridge the curve had a bottom in the middle; here the strongest penalty is the best, and it is still not enough to beat one column.

</details>

---

### G9 · How much coefficient is left  ★★★☆☆  · revisits S6

For C in `[100, 1, 0.01, 0.0001]`, fit the pipeline on all columns and compute the sum of the squared coefficients, the quantity the penalty charges for. Print it for each C, without a loop over the coefficients.

In [ ]:
for C in [100, 1, 0.01, 0.0001]:
    fitted_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=C, max_iter=1000))])
    fitted_pipe.fit(train[columns], train['rising'])
    size = ...
    print(C, size)

<details>
<summary>💡 Hint</summary>

`fitted_pipe.named_steps['logit'].coef_` is an array with one row; square it and sum it: `(coef_ ** 2).sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
for C in [100, 1, 0.01, 0.0001]:
    fitted_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=C, max_iter=1000))])
    fitted_pipe.fit(train[columns], train['rising'])
    size = (fitted_pipe.named_steps['logit'].coef_ ** 2).sum()
    print(C, round(size, 6))
```

12.38 at C = 100, 10.43 at 1, 0.6209 at 0.01 and 0.003065 at 0.0001. The penalty term shrinks by a factor of about a hundred for every factor of a hundred in C, and you can watch it go, as you did for alpha.

</details>

---

### G10 · The test rows would choose differently  ★★★★☆  · revisits S5

For each C in `[0.0001, 0.001, 0.01, 0.1, 1, 10]`, cross-validate the pipeline on the training rows **and** fit it and score it on the test rows, storing the two AUCs in two dictionaries. Print the C the folds pick and the C the test rows would pick. Which one is allowed to choose?

In [ ]:
cv_by_C = {}
test_by_C = {}

for C in [0.0001, 0.001, 0.01, 0.1, 1, 10]:
    ...

print(test_by_C)
print('the folds pick          :', ...)
print('the test rows would pick:', ...)

<details>
<summary>💡 Hint 1</summary>

Inside the loop, `cross_val_score(...)` on the training rows for the first dictionary, then `candidate.fit(train[columns], train['rising'])` and `roc_auc_score` on `candidate.predict_proba(test[columns])[:, 1]` for the second.

</details>

<details>
<summary>💡 Hint 2</summary>

`max(cv_by_C, key=cv_by_C.get)` and `max(test_by_C, key=test_by_C.get)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
cv_by_C = {}
test_by_C = {}

for C in [0.0001, 0.001, 0.01, 0.1, 1, 10]:
    candidate = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=C, max_iter=1000))])
    candidate_scores = cross_val_score(candidate, train[columns], train['rising'], cv=folds, scoring='roc_auc')
    cv_by_C[C] = round(float(candidate_scores.mean()), 4)
    candidate.fit(train[columns], train['rising'])
    test_by_C[C] = round(float(roc_auc_score(test['rising'], candidate.predict_proba(test[columns])[:, 1])), 4)

print(test_by_C)
print('the folds pick          :', max(cv_by_C, key=cv_by_C.get))
print('the test rows would pick:', max(test_by_C, key=test_by_C.get))
```

The test rows would pick C = 10 at 0.818, and the folds picked 0.0001, which scores 0.811 on the test rows. The two disagree, and only the folds are allowed to choose: a C picked on the test rows leaves nothing to report it with. The test AUCs are within 0.04 of each other, which is the size of the disagreement.

</details>

---

## ✂️ H · The l1 penalty, and the arguments

Exact zeros, the settings, and the error you will meet. H3 and H4 share nothing but the lesson.

### H1 · The lasso's penalty on a classifier  ★★☆☆☆

Build and fit a pipeline with `LogisticRegression(penalty='l1', solver='liblinear', C=0.01)` on all columns, and print the names of the columns whose coefficient is not zero, with the coefficient.

In [ ]:
sparse = Pipeline([...])
...

...

<details>
<summary>💡 Hint 1</summary>

Name the step `'logit'`. The coefficients are `sparse.named_steps['logit'].coef_[0]`, one row.

</details>

<details>
<summary>💡 Hint 2</summary>

`for name, b in zip(columns, coefs):` with `if b != 0: print(name, round(b, 3))`.

</details>

<details>
<summary>✅ Solution</summary>

```python
sparse = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(penalty='l1', solver='liblinear', C=0.01))])
sparse.fit(train[columns], train['rising'])

for name, b in zip(columns, sparse.named_steps['logit'].coef_[0]):
    if b != 0:
        print(name, round(b, 3))
```

`vol_20d` and `ret_5d`, and nothing else: 17 of the 19 coefficients are exactly zero. The one column the lecture started with is one of the two the penalty keeps.

</details>

---

### H2 · How many survive as C grows  ★★★☆☆  · revisits S2

For C in `[0.001, 0.003, 0.01, 0.03, 0.1, 1]`, fit the l1 pipeline and store the number of non-zero coefficients in a dictionary. Print it.

In [ ]:
survivors = {}

for C in [0.001, 0.003, 0.01, 0.03, 0.1, 1]:
    ...

print(survivors)

<details>
<summary>💡 Hint</summary>

`int((candidate.named_steps['logit'].coef_[0] != 0).sum())` counts the survivors as a plain number.

</details>

<details>
<summary>✅ Solution</summary>

```python
survivors = {}

for C in [0.001, 0.003, 0.01, 0.03, 0.1, 1]:
    candidate = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(penalty='l1', solver='liblinear', C=C))])
    candidate.fit(train[columns], train['rising'])
    survivors[C] = int((candidate.named_steps['logit'].coef_[0] != 0).sum())

print(survivors)
```

0 at C = 0.001, then 0, 2, 6, 13 and 18 at C = 1. With the l1 penalty a small C removes columns rather than merely shrinking them, and at C = 0.001 the model is the intercept alone.

</details>

---

### H3 · What the defaults are  ★☆☆☆☆

Print the `penalty`, `C`, `solver` and `max_iter` of a `LogisticRegression()` from `get_params()`.

In [ ]:
settings = ...
print(..., ..., ..., ...)

<details>
<summary>💡 Hint</summary>

`get_params()` returns a dictionary; read four keys from it.

</details>

<details>
<summary>✅ Solution</summary>

```python
settings = LogisticRegression().get_params()
print(settings['penalty'], settings['C'], settings['solver'], settings['max_iter'])
```

`l2`, `1.0`, `lbfgs` and `100`. Every logistic regression in this notebook that did not say otherwise had a ridge penalty of strength 1 on, solved by lbfgs in at most 100 steps.

</details>

---

### H4 · The error you will meet  ★★★☆☆  · revisits S1

The cell below asks the default solver for the l1 penalty, and it raises. Run it and read the message, then fix the call in the second cell, fit it, and print the number of non-zero coefficients.

In [ ]:
broken = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(penalty='l1', C=0.01))])
broken.fit(train[columns], train['rising'])

In [ ]:
fixed = ...
...
print(...)

<details>
<summary>💡 Hint 1</summary>

The message names the solvers that can handle `l1`.

</details>

<details>
<summary>💡 Hint 2</summary>

`solver='liblinear'` is the one the lecture used.

</details>

<details>
<summary>✅ Solution</summary>

```python
fixed = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(penalty='l1', solver='liblinear', C=0.01))])
fixed.fit(train[columns], train['rising'])
print((fixed.named_steps['logit'].coef_[0] != 0).sum())
```

A `ValueError` saying that lbfgs supports only the l2 penalty, then 2 once the solver is changed. The last line of a traceback names the problem, and here it also names the cure.

</details>

---

## 📖 I · Reading the result

What the numbers mean, and which model to report. Each of these fits what it needs.

### I1 · A coefficient per standard deviation, in a sentence  ★★☆☆☆  · revisits S1

Fit a scaling pipeline with `LogisticRegression()` on `vol_20d` alone, take the coefficient, and print one sentence with an f-string: the factor by which one standard deviation more volatility multiplies the odds of a rise, to two decimals.

In [ ]:
one_pipe = ...
...
b = ...

sentence = ...
print(sentence)

<details>
<summary>💡 Hint</summary>

`one_pipe.named_steps['logit'].coef_[0, 0]` is the coefficient per standard deviation; `np.exp(b)` is the factor.

</details>

<details>
<summary>✅ Solution</summary>

```python
one_pipe = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression())])
one_pipe.fit(train[['vol_20d']], train['rising'])
b = one_pipe.named_steps['logit'].coef_[0, 0]

sentence = f'One standard deviation more volatility multiplies the odds of a rise by {np.exp(b):.2f}.'
print(sentence)
```

The coefficient is -0.934 per standard deviation and the factor is 0.39. Per percentage point the factor was 0.26; the two describe the same curve in different units of the column.

</details>

---

### I2 · Four test AUCs, ranked  ★★★☆☆  · revisits S2

Put the test AUC of four models into one dictionary: one column; 19 columns in a scaling pipeline with C = 1; 19 columns with C = 0.0001; and 19 columns with the l1 penalty at C = 0.01. Print them from highest to lowest.

In [ ]:
four = {}

...

for name in sorted(four, key=four.get, reverse=True):
    print(f'{name:22} {four[name]:.4f}')

<details>
<summary>💡 Hint 1</summary>

Each entry is one model fitted on the training rows and scored with `roc_auc_score` on the test rows.

</details>

<details>
<summary>💡 Hint 2</summary>

`sorted(four, key=four.get, reverse=True)` orders the keys from the largest value down.

</details>

<details>
<summary>✅ Solution</summary>

```python
four = {}

one = LogisticRegression()
one.fit(train[['vol_20d']], train['rising'])
four['one column'] = roc_auc_score(test['rising'], one.predict_proba(test[['vol_20d']])[:, 1])

wide_1 = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=1, max_iter=1000))])
wide_1.fit(train[columns], train['rising'])
four['19 columns, C = 1'] = roc_auc_score(test['rising'], wide_1.predict_proba(test[columns])[:, 1])

wide_small = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(C=0.0001, max_iter=1000))])
wide_small.fit(train[columns], train['rising'])
four['19 columns, C chosen'] = roc_auc_score(test['rising'], wide_small.predict_proba(test[columns])[:, 1])

wide_l1 = Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(penalty='l1', solver='liblinear', C=0.01))])
wide_l1.fit(train[columns], train['rising'])
four['19 columns, l1'] = roc_auc_score(test['rising'], wide_l1.predict_proba(test[columns])[:, 1])

for name in sorted(four, key=four.get, reverse=True):
    print(f'{name:22} {four[name]:.4f}')
```

Highest is "19 columns, l1" at 0.8153 and lowest "one column" at 0.8075, all four within 0.02. On the folds the one-column model was ahead by a wide margin; on the test years the four are level. That is the fold spread from the lecture showing up as a reminder that two years of test rows are two years.

</details>

---

### I3 · One function for any classifier  ★★★★☆  · revisits S2

Write `evaluate(p)`: it cross-validates the model or pipeline `p` on the training rows with `folds` and the AUC, refits it on all of them, and returns the pair `(cv_auc, test_auc)`. Run it on the one-column model with `train[['vol_20d']]` and `test[['vol_20d']]`, so give the function the two feature tables as arguments too.

In [ ]:
def evaluate(p, X_train, X_test):
    ...

print('one column:', evaluate(LogisticRegression(), train[['vol_20d']], test[['vol_20d']]))
print('19 columns:', evaluate(Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))]), train[columns], test[columns]))

<details>
<summary>💡 Hint 1</summary>

Inside: `cross_val_score(p, X_train, train['rising'], cv=folds, scoring='roc_auc')` for the first number, then `p.fit(X_train, train['rising'])` and `roc_auc_score` on `p.predict_proba(X_test)[:, 1]` for the second.

</details>

<details>
<summary>💡 Hint 2</summary>

`return round(float(cv.mean()), 4), round(test_auc, 4)` returns a pair.

</details>

<details>
<summary>✅ Solution</summary>

```python
def evaluate(p, X_train, X_test):
    cv = cross_val_score(p, X_train, train['rising'], cv=folds, scoring='roc_auc')
    p.fit(X_train, train['rising'])
    test_auc = roc_auc_score(test['rising'], p.predict_proba(X_test)[:, 1])
    return round(float(cv.mean()), 4), round(float(test_auc), 4)

print('one column:', evaluate(LogisticRegression(), train[['vol_20d']], test[['vol_20d']]))
print('19 columns:', evaluate(Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))]), train[columns], test[columns]))
```

One column (0.7826, 0.8075) and 19 columns (0.6787, 0.8152). Because every scikit-learn classifier fits and gives probabilities the same way, the function never needs to know what is inside it.

</details>

---

### I4 · The rise the model was surest would not come  ★★★☆☆  · revisits S5

Fit the one-column model, take the test probabilities, and among the days **with** a rise find the one given the lowest probability. Print its date, its probability and its volatility.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

p_rise = ...
worst = ...
print(..., ..., ...)

<details>
<summary>💡 Hint 1</summary>

`p_rise = np.where(rose, p, 9)` keeps the probability on the days with a rise and puts a 9 everywhere else, so `p_rise.argmin()` is the position of the rise with the lowest probability.

</details>

<details>
<summary>💡 Hint 2</summary>

`test.index[worst].date()`, `p[worst]` and `test['vol_20d'].iloc[worst]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

p_rise = np.where(rose, p, 9)
worst = p_rise.argmin()
print(test.index[worst].date(), round(p[worst], 3), round(test['vol_20d'].iloc[worst], 3))
```

2023-02-23, given a probability of 0.437 at a volatility of 1.06 percent. Look up what the market did in the 20 days after that date. A model that only sees the last 20 days cannot see the event coming, and the rise it was surest would not come is always the one before an event.

</details>

---

### I5 · Where the two kinds of day sit  ★★★☆☆  · revisits S3

Draw two histograms on one axis: the test probabilities of the days with a rise and of the days without, with a legend, and a vertical line at 0.5.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.hist(p[rose], bins=30, alpha=0.6, label='a rise came')` and the same for `p[~rose]`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.axvline(0.5, color='black')` and `ax.legend()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]
rose = test['rising'].values == 1

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.hist(p[rose], bins=30, alpha=0.6, label='a rise came')
ax.hist(p[~rose], bins=30, alpha=0.6, label='no rise')
ax.axvline(0.5, color='black')
ax.set_xlabel('probability of a rise')
ax.set_ylabel('test days')
ax.legend()
plt.show()
```

The two piles overlap across the whole range, and the days with a rise sit further right. Every threshold cuts through both piles, which is the trade-off between the two kinds of error drawn as data.

</details>

---

## 🔁 J · Across the desk

The whole workflow, once per instrument. J2 to J4 use the function from J1; if you skipped J1, copy its solution into the first cell.

### J1 · A table with a label, for any ticker  ★★★★☆  · revisits S6

Write `build_table(ticker)`: the six volatility windows `[5, 10, 20, 40, 60, 120]` and the three return windows `[5, 20, 60]` of that ticker, the 20-day volatility of every **other** instrument as `<ticker>_vol`, the target `vol_next`, incomplete rows dropped, and then the label `rising`. Return the table, and check its shape on `'SPY'`.

In [ ]:
def build_table(ticker):
    ...

spy_table = build_table('SPY')
print(spy_table)

<details>
<summary>💡 Hint 1</summary>

Three loops, as when the table was built for ridge: `'vol_' + str(w) + 'd'` with `.rolling(w).std()`, `'ret_' + str(w) + 'd'` with `.rolling(w).mean()`, and `if t != ticker:` inside the loop over `rets.columns`.

</details>

<details>
<summary>💡 Hint 2</summary>

Add `vol_next` as `.rolling(20).std().shift(-20)`, `dropna()`, and only then the label, so the comparison sees complete rows.

</details>

<details>
<summary>✅ Solution</summary>

```python
def build_table(ticker):
    frame = pd.DataFrame()
    for w in [5, 10, 20, 40, 60, 120]:
        frame['vol_' + str(w) + 'd'] = rets[ticker].rolling(w).std()
    for w in [5, 20, 60]:
        frame['ret_' + str(w) + 'd'] = rets[ticker].rolling(w).mean()
    for t in rets.columns:
        if t != ticker:
            frame[t + '_vol'] = rets[t].rolling(20).std()
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()
    frame['rising'] = (frame['vol_next'] > frame['vol_20d']).astype(int)
    return frame

print(build_table('SPY').shape)
```

2,376 rows and 21 columns: 19 features, the target and the label, which for SPY is the setup's table. The label goes on last, after `dropna()`, so it never compares an incomplete row.

</details>

---

### J2 · Nvidia  ★★★☆☆

Build the table for `'NVDA'`, split at the end of 2022, fit the one-column logistic regression on `vol_20d`, and print its test AUC, its test accuracy, and the majority rule's accuracy on the same rows.

In [ ]:
nvda = build_table('NVDA')
tr = ...
te = ...

model_n = LogisticRegression()
...

print('AUC     :', ...)
print('accuracy:', ...)
print('majority:', ...)

<details>
<summary>💡 Hint</summary>

The three fits and scores from earlier sections with `tr` and `te` in place of `train` and `test`. The majority rule is the larger of the share of rises and one minus it.

</details>

<details>
<summary>✅ Solution</summary>

```python
nvda = build_table('NVDA')
tr = nvda.loc[:'2022-12-31']
te = nvda.loc['2023-01-01':]

model_n = LogisticRegression()
model_n.fit(tr[['vol_20d']], tr['rising'])

print('AUC     :', roc_auc_score(te['rising'], model_n.predict_proba(te[['vol_20d']])[:, 1]))
print('accuracy:', accuracy_score(te['rising'], model_n.predict(te[['vol_20d']])))
print('majority:', max(te['rising'].mean(), 1 - te['rising'].mean()))
```

An AUC of 0.831 and an accuracy of 0.703 against 0.512 for the majority rule. Nvidia's curve crosses one half at 2.65 percent, three times the index's, because its volatility is three times the index's; the label has no units, but the crossing point does.

</details>

---

### J3 · Every instrument's AUC  ★★★★☆  · revisits S3

For every ticker in `rets`, build its table, fit the one-column model on the training rows and store the test AUC in a dictionary `desk_auc`. Draw the AUCs as a bar chart, sorted from highest to lowest, with a line at 0.5.

In [ ]:
desk_auc = {}
for ticker in rets.columns:
    ...

ranked = ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

The loop body is J2 with `ticker` in place of `'NVDA'`, ending in `desk_auc[ticker] = roc_auc_score(...)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ranked = pd.Series(desk_auc).sort_values(ascending=False)`, then `ax.bar(ranked.index, ranked.values)` and `ax.axhline(0.5)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
desk_auc = {}
for ticker in rets.columns:
    frame = build_table(ticker)
    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]
    model_t = LogisticRegression()
    model_t.fit(tr[['vol_20d']], tr['rising'])
    desk_auc[ticker] = roc_auc_score(te['rising'], model_t.predict_proba(te[['vol_20d']])[:, 1])

ranked = pd.Series(desk_auc).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(ranked.index, ranked.values)
ax.axhline(0.5, color='grey', linestyle='--')
ax.set_ylabel('test AUC')
ax.set_title('One column, logistic regression: test AUC per instrument', loc='left')
plt.show()
```

From 0.896 on DIS down to 0.739 on MSFT, every one of the 11 well above one half. Whether next month is more volatile than this one is predictable from this month alone on every instrument, because volatility mean-reverts everywhere.

</details>

---

### J4 · Which C does each instrument pick  ★★★★★  · revisits S2

For every ticker, build its table, run the grid search from G5 on its 19 columns and training rows, and store the winning C in a dictionary `best_C`. Print it, and count how many instruments pick the smallest value in the grid.

In [ ]:
best_C = {}

for ticker in rets.columns:
    ...

print(best_C)
print('pick 0.0001:', ...)

<details>
<summary>💡 Hint 1</summary>

The feature columns of a built table are `list(frame.columns[:-2])`: every column but the target and the label.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for t in best_C if best_C[t] == 0.0001)` counts. Eleven searches take a little while.

</details>

<details>
<summary>✅ Solution</summary>

```python
best_C = {}

for ticker in rets.columns:
    frame = build_table(ticker)
    tr = frame.loc[:'2022-12-31']
    cols = list(frame.columns[:-2])
    search_t = GridSearchCV(Pipeline([('scale', StandardScaler()), ('logit', LogisticRegression(max_iter=1000))]),
                            {'logit__C': [0.0001, 0.001, 0.01, 0.1, 1, 10]},
                            cv=folds, scoring='roc_auc')
    search_t.fit(tr[cols], tr['rising'])
    best_C[ticker] = search_t.best_params_['logit__C']

print(best_C)
print('pick 0.0001:', sum(1 for t in best_C if best_C[t] == 0.0001))
```

5 of 11 pick 0.0001; JNJ picks 10, KO picks 1, MSFT picks 0.01, NVDA picks 0.1, PG picks 0.01, WMT picks 0.1. For ridge, every instrument agreed on alpha; here the chosen C runs across the whole grid. On a label the folds are noisier than on a number, because a 0 or a 1 carries less information than a volatility, so the choice of C moves with the instrument. The test rows were never touched.

</details>

---

## 🧩 K · Small cases

Five short investigations that each start from scratch: no variable from an earlier section, and no scaffold beyond the first line. Each one needs today's tools and a few older ones.

### K1 · A month of predictions, checked one by one  ★★★☆☆  · revisits S2

Fit the one-column model and take the test probabilities. For the **last 20** test days, print one line each: the date, the probability to two decimals, the word `rise` or `no rise` depending on whether the probability is at least one half, and `right` or `wrong` depending on the label. Count the days the model got right.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]

hits = 0
...

print('right on', hits, 'of 20 days')

<details>
<summary>💡 Hint 1</summary>

`zip(test.index[-20:], p[-20:], test['rising'].values[-20:])` walks the three together. `date.date()` prints a date without the time.

</details>

<details>
<summary>💡 Hint 2</summary>

Inside the loop, an `if` sets `call` to 1 or 0 and the word; `right` is `call == label`; add one to `hits` when it is.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
p = model.predict_proba(test[['vol_20d']])[:, 1]

hits = 0
for date, prob, label in zip(test.index[-20:], p[-20:], test['rising'].values[-20:]):
    if prob >= 0.5:
        call, word = 1, 'rise'
    else:
        call, word = 0, 'no rise'
    if call == label:
        verdict = 'right'
        hits = hits + 1
    else:
        verdict = 'wrong'
    print(f'{date.date()}  {prob:.2f}  {word:8}  {verdict}')

print('right on', hits, 'of 20 days')
```

Right on 15 of the last 20 test days. Twenty lines of a loop with an `if` inside, an f-string with a width for the word so the columns line up, and a counter: the pieces of the first two sessions, doing the reading that the confusion matrix does in one line.

</details>

---

### K2 · Which instrument rose most often in 2024  ★★★☆☆  · revisits S3

For every ticker in `rets`, compute the share of 2024 days after which volatility rose, straight from the returns: the 20-day volatility, the same series shifted 20 days up, a table of the two with incomplete rows dropped, and the comparison. Store the shares in a dictionary and print the three largest.

In [ ]:
share_2024 = {}

for ticker in rets.columns:
    ...

for ticker in sorted(share_2024, key=share_2024.get, reverse=True)[:3]:
    print(ticker, share_2024[ticker])

<details>
<summary>💡 Hint 1</summary>

`now = rets[ticker].rolling(20).std()`, `after = now.shift(-20)`, then `both = pd.DataFrame({'now': now, 'after': after}).dropna()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`(both['after'] > both['now']).loc['2024-01-01':'2024-12-31'].mean()` is the share; wrap it in `round(float(...), 3)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
share_2024 = {}

for ticker in rets.columns:
    now = rets[ticker].rolling(20).std()
    after = now.shift(-20)
    both = pd.DataFrame({'now': now, 'after': after}).dropna()
    share_2024[ticker] = round(float((both['after'] > both['now']).loc['2024-01-01':'2024-12-31'].mean()), 3)

for ticker in sorted(share_2024, key=share_2024.get, reverse=True)[:3]:
    print(ticker, share_2024[ticker])
```

DIS at 0.565, then KO at 0.543 and JPM at 0.53. The `dropna()` matters: the last 20 days of 2024 have no next month, and a comparison with a missing value quietly counts as 0.

</details>

---

### K3 · One sentence per instrument  ★★★★☆  · revisits S2

Write `report(ticker)`: from the returns alone, build a two-column table of `vol_20d` and `vol_next` for that ticker, add the label, split at the end of 2022, fit the one-column logistic regression, and **return** one sentence with the test AUC to two decimals and the test accuracy next to the majority rule, both as percentages. Print it for `KO`, `JPM` and `DIS`.

In [ ]:
def report(ticker):
    ...

for ticker in ['KO', 'JPM', 'DIS']:
    print(report(ticker))

<details>
<summary>💡 Hint 1</summary>

Inside: `now = rets[ticker].rolling(20).std()`, the table `pd.DataFrame({'vol_20d': now, 'vol_next': now.shift(-20)}).dropna()`, the label as a comparison, then the split and the fit as in section E.

</details>

<details>
<summary>💡 Hint 2</summary>

`f'{ticker}: AUC {auc:.2f}, accuracy {acc:.1%} against {majority:.1%} for the majority rule.'`

</details>

<details>
<summary>✅ Solution</summary>

```python
def report(ticker):
    now = rets[ticker].rolling(20).std()
    frame = pd.DataFrame({'vol_20d': now, 'vol_next': now.shift(-20)}).dropna()
    frame['rising'] = (frame['vol_next'] > frame['vol_20d']).astype(int)
    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]
    m = LogisticRegression()
    m.fit(tr[['vol_20d']], tr['rising'])
    auc = roc_auc_score(te['rising'], m.predict_proba(te[['vol_20d']])[:, 1])
    acc = accuracy_score(te['rising'], m.predict(te[['vol_20d']]))
    majority = max(te['rising'].mean(), 1 - te['rising'].mean())
    return f'{ticker}: AUC {auc:.2f}, accuracy {acc:.1%} against {majority:.1%} for the majority rule.'

for ticker in ['KO', 'JPM', 'DIS']:
    print(report(ticker))
```

Coca-Cola scores an AUC of 0.85, JPMorgan 0.79 and Disney 0.90, each with an accuracy above its majority rule. The whole workflow, from raw returns to a scored classifier, is one function of twelve lines, and the sentence it returns is the one a report would carry.

</details>

---

### K4 · The longest run of predicted rises  ★★★★☆  · revisits S2

Fit the one-column model and predict the test rows. Walk through the predictions in order with a loop and find the longest run of consecutive days predicted as a rise, and the date on which that run ended.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

streak = 0
longest = 0
end_date = None
...

print(longest, 'days, ending', end_date)

<details>
<summary>💡 Hint 1</summary>

`for date, value in zip(test.index, predicted):` and, inside, add one to `streak` when `value == 1` and set it back to 0 otherwise.

</details>

<details>
<summary>💡 Hint 2</summary>

Whenever `streak` passes `longest`, store both `streak` and `date.date()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

streak = 0
longest = 0
end_date = None
for date, value in zip(test.index, predicted):
    if value == 1:
        streak = streak + 1
        if streak > longest:
            longest = streak
            end_date = date.date()
    else:
        streak = 0

print(longest, 'days, ending', end_date)
```

108 trading days, ending 2023-10-27. A run that long is the model saying "volatility will rise" through a whole stretch of calm, because volatility stayed below the crossing point the entire time. A counter that resets is the loop shape for any "longest run" question.

</details>

---

### K5 · Accuracy, year by year  ★★★☆☆  · revisits S3

Fit the one-column model, predict the test rows, and compute the accuracy separately for 2023 and 2024 with `groupby`: make a Series of `True` and `False` for whether each prediction was right, indexed by date, and group it by year.

In [ ]:
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

right = ...
by_year = ...
print(by_year)

<details>
<summary>💡 Hint 1</summary>

`right = pd.Series(predicted == test['rising'].values, index=test.index)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`right.groupby(right.index.year).mean()`: the mean of a column of `True` and `False` is the share of `True`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LogisticRegression()
model.fit(train[['vol_20d']], train['rising'])
predicted = model.predict(test[['vol_20d']])

right = pd.Series(predicted == test['rising'].values, index=test.index)
by_year = right.groupby(right.index.year).mean()
print(by_year)
```

0.688 in 2023 and 0.698 in 2024, close to each other: one accuracy over two years hid nothing this time, and a `groupby` on the year is the one-line way to check that.

</details>

---

## 🏁 Done

You made labels, watched a straight line give probabilities below zero, fitted the curve that cannot, and read its two numbers as a crossing point and an odds factor. You turned probabilities into predictions at a threshold of your choosing, scored them with the four counts and the AUC, chose C on the folds, and ran the whole workflow on every instrument.

The case takes the same tools back to the risk report, where the data is in decimals, and where the default penalty does something to a raw column that the label's lack of units does not protect against.